# 04 — Stage D: Packaging, Deployment and Monitoring

**Step 11 of the 11-step predictive-modeling process.**

**Prerequisites:** notebooks 01–03, in order. This one reads `data/processed/clean.parquet`
and `artifacts/split_manifest.json`.

A model that exists only inside a notebook is not a deliverable. This notebook turns it
into an artifact someone else can load, plus the monitoring that tells them when it has
stopped working.

| Section | What it produces |
|---|---|
| 11.1 | The final pipeline refitted and saved to `artifacts/model.joblib` |
| 11.2 | A model card — training window, metrics, versions, split hash |
| 11.3 | **Round-trip proof**: reload from disk, assert identical predictions |
| 11.4 | Inference demo on raw-shaped input |
| 11.5 | **PSI** drift monitoring, per feature |
| 11.6 | Performance decay and the retraining policy |
| 11.7 | The process's 6-question self-check, answered |

> *The world changes — data drift and concept drift — so the model degrades. Plan for
> retraining.* Sections 11.5 and 11.6 are that plan, measured rather than asserted.

## 11.0 Setup — rebuild the pipeline from notebooks 02 and 03

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn

from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PROCESSED = ROOT / "data" / "processed"
ARTIFACTS = ROOT / "artifacts"
FIG_DIR = ARTIFACTS / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

C_BLUE, C_ORANGE, C_AQUA, C_RED, C_GRAY = "#2a78d6", "#eb6834", "#1baf7a", "#e34948", "#8a8a86"
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110, "savefig.dpi": 150,
                     "savefig.bbox": "tight", "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 12, "axes.titleweight": "bold"})

def save_fig(name):
    plt.savefig(FIG_DIR / f"{name}.png")
    plt.show()

df = pd.read_parquet(DATA_PROCESSED / "clean.parquet")
manifest = json.loads((ARTIFACTS / "split_manifest.json").read_text())
CUTOFF_DATE = pd.Timestamp(manifest["cutoff_date"])
DATA_SOURCE = manifest.get("data_source", "unknown")

NON_FEATURES = ["target", "is_train", "is_train_random_ref", "issue_d",
                "earliest_cr_line", "grade", "sub_grade"]
feature_cols = [c for c in df.columns if c not in NON_FEATURES]
X, y = df[feature_cols], df["target"]
train_mask = df["is_train"].to_numpy()
X_train, X_test = X.loc[train_mask], X.loc[~train_mask]
y_train, y_test = y.loc[train_mask], y.loc[~train_mask]

assert df.loc[train_mask, "issue_d"].max() < df.loc[~train_mask, "issue_d"].min(), \
    "Look-ahead: train and test issue_d ranges overlap."
print(f"Loaded {len(df):,} rows | source: {DATA_SOURCE} | cutoff {CUTOFF_DATE.date()}")
print(f"train {len(X_train):,} / test {len(X_test):,} | {len(feature_cols)} features")

## 11.1 Refit and save the complete pipeline

Two decisions worth stating explicitly.

**What gets saved.** The whole `Pipeline` — imputation, scaling, encoding, the estimator
and the calibration wrapper — not just the fitted estimator. Saving the estimator alone is
the classic deployment bug: production then has to re-implement the preprocessing, and any
discrepancy silently corrupts every prediction. One object in, one object out.

**What it is fitted on.** The **full training period**, including the calibration slice
that notebook 03 held out — more data for the final model, and legitimate because the
calibration slice was always training-period data. The **test period is still never
touched**. Its metrics remain the honest out-of-sample estimate precisely because the
deployed model never saw it.

The model chosen here is the interpretable baseline. Notebook 03 found the ensembles
overfit heavily (large train-vs-CV gaps) while logistic regression generalised best — and
in credit scoring an auditable, monotone, easily-monitored model is worth real money in
governance terms.

In [ ]:
numeric_cols = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

preprocessor = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("encode", OneHotEncoder(handle_unknown="infrequent_if_exist",
                                               min_frequency=0.01,
                                               sparse_output=False))]), categorical_cols),
])

FINAL_MODEL_NAME = "Logistic Regression"
base_pipeline = Pipeline([
    ("prep", preprocessor),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=2000,
                               random_state=RANDOM_SEED)),
])

# Time-ordered inner split so calibration still fits on data LATER than the model fit.
issue_train = df.loc[train_mask, "issue_d"]
order = np.argsort(issue_train.to_numpy(), kind="mergesort")
n_fit = int(len(order) * 0.8)
fit_idx, cal_idx = order[:n_fit], order[n_fit:]

base_pipeline.fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])
try:
    from sklearn.frozen import FrozenEstimator
    final_model = CalibratedClassifierCV(FrozenEstimator(base_pipeline), method="isotonic")
except ImportError:
    final_model = CalibratedClassifierCV(base_pipeline, method="isotonic", cv="prefit")
final_model.fit(X_train.iloc[cal_idx], y_train.iloc[cal_idx])

test_scores = final_model.predict_proba(X_test)[:, 1]
final_roc = roc_auc_score(y_test, test_scores)
final_pr = average_precision_score(y_test, test_scores)
print(f"Final model: {FINAL_MODEL_NAME} (calibrated)")
print(f"  OOT ROC-AUC {final_roc:.4f} | PR-AUC {final_pr:.4f}")

MODEL_PATH = ARTIFACTS / "model.joblib"
joblib.dump(final_model, MODEL_PATH)
print(f"\nSaved complete pipeline -> {MODEL_PATH} "
      f"({MODEL_PATH.stat().st_size / 1024:.0f} KB)")

## 11.2 The model card

Everything a future reader needs to answer *"what is this file, and can I trust it?"*
without rerunning anything. The library versions matter more than they look: an unpickled
scikit-learn model loaded under a different version can fail outright or, worse, load and
behave differently.

In [ ]:
model_card = {
    "name": "lending-club-credit-scoring",
    "model": FINAL_MODEL_NAME + " + isotonic calibration",
    "created_utc": pd.Timestamp.utcnow().isoformat(),
    "data_source": DATA_SOURCE,
    "task": "Binary classification: P(charged off) at application time",
    "target": "1 = Charged Off/Default, 0 = Fully Paid (terminal statuses only)",
    "split": {
        "type": manifest["split_type"],
        "cutoff_date": manifest["cutoff_date"],
        "train_window": manifest["train_date_range"],
        "test_window": manifest["test_date_range"],
        "test_index_md5": manifest["test_index_md5"],
    },
    "training_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "train_default_rate": float(y_train.mean()),
    "test_default_rate": float(y_test.mean()),
    "metrics_out_of_time": {"roc_auc": float(final_roc), "pr_auc": float(final_pr)},
    "n_features": len(feature_cols),
    "features": feature_cols,
    "random_seed": RANDOM_SEED,
    "versions": {
        "python": sys.version.split()[0],
        "scikit-learn": sklearn.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "joblib": joblib.__version__,
    },
    "known_limitations": [
        "issue_d is a proxy for application date",
        "Right-censoring: terminal-status filter drops unmatured loans",
        "Accepted applicants only: risk conditional on acceptance (reject inference)",
        "Fairness screened on proxies only; no protected attributes in the data",
        "Cost ratio FN:FP = 4:1 is a placeholder, not portfolio economics",
    ],
}
(ARTIFACTS / "model_card.json").write_text(json.dumps(model_card, indent=2))
print(json.dumps({k: v for k, v in model_card.items() if k != "features"}, indent=2))

## 11.3 Round-trip proof

Writing a file is not the same as writing a *usable* file. This cell reloads the pipeline
from disk in a fresh object and asserts its predictions are **bit-for-bit identical** to
the in-memory model.

This is the cell that would catch an unpicklable custom transformer, a lost preprocessing
step, or a version mismatch — the failures that otherwise surface in production rather than
here.

In [ ]:
reloaded = joblib.load(MODEL_PATH)
reloaded_scores = reloaded.predict_proba(X_test)[:, 1]

max_diff = float(np.max(np.abs(reloaded_scores - test_scores)))
identical = np.array_equal(reloaded_scores, test_scores)
print(f"max |reloaded - in-memory| = {max_diff:.2e}")
print(f"bit-for-bit identical      : {identical}")
assert identical, "Round-trip FAILED: the saved pipeline does not reproduce predictions."
print("\nOK -- the artifact on disk is the model that was evaluated.")

## 11.4 Inference demo

The deployment path, end to end: take raw-shaped rows, hand them to the reloaded pipeline,
get calibrated probabilities. No preprocessing code outside the artifact — which is the
whole point of 11.1.

In [ ]:
sample = X_test.head(5).copy()
probabilities = reloaded.predict_proba(sample)[:, 1]

OPERATING_THRESHOLD = 0.195     # from notebook 03 section 9.3 (cost-based)
demo = pd.DataFrame({
    "loan_amnt": sample["loan_amnt"].values,
    "annual_inc": sample["annual_inc"].values,
    "dti": sample["dti"].values,
    "fico_avg": sample["fico_avg"].values,
    "P(default)": probabilities,
    "decision": np.where(probabilities >= OPERATING_THRESHOLD, "DECLINE", "APPROVE"),
    "actual": np.where(y_test.head(5).values == 1, "Charged Off", "Fully Paid"),
})
display(demo.style.format({"loan_amnt": "{:,.0f}", "annual_inc": "{:,.0f}", "dti": "{:.1f}",
                           "fico_avg": "{:.0f}", "P(default)": "{:.3f}"})
        .set_caption("Table 13 - Inference demo: raw rows in, calibrated decisions out"))

**Table 13 — Inference demo.**

Five applications scored through the artifact exactly as production would. The `actual`
column is shown for orientation only — it is not available at scoring time, which is the
entire premise of the project.

**Action →** This is the contract a serving layer implements: load `model.joblib`, pass a
DataFrame with the columns listed in the model card, apply the threshold. Any preprocessing
written outside the artifact is a bug.

## 11.5 Monitoring — population stability index (PSI)

> *The world changes, so the model degrades.*

**PSI** is the standard drift measure in credit risk. It compares the distribution of a
feature between the period the model was trained on and the period being scored:

`PSI = Σ (actual% − expected%) × ln(actual% / expected%)`

Conventional thresholds:

| PSI | Reading | Action |
|---|---|---|
| < 0.10 | Stable | None |
| 0.10 – 0.25 | Moderate shift | Investigate |
| > 0.25 | Major shift | Retrain |

Here the training period is the baseline and the test period is the "live" population, so
this measures the drift the model has *already* been evaluated across. In production the
same function runs with the most recent scoring window in place of the test set.

PSI measures **data drift** — the inputs moving. It does not detect **concept drift**, where
the inputs look the same but their relationship to default has changed. That needs outcome
labels, which arrive months later; 11.6 covers it.

In [ ]:
def population_stability_index(expected, actual, n_bins: int = 10) -> float:
    """PSI between a baseline (expected) and a current (actual) sample."""
    expected, actual = pd.Series(expected).dropna(), pd.Series(actual).dropna()
    if expected.empty or actual.empty:
        return np.nan
    # Quantile edges from the BASELINE only -- the live period must not define its own bins.
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, n_bins + 1)))
    if len(edges) < 3:
        return np.nan
    edges[0], edges[-1] = -np.inf, np.inf
    e = np.histogram(expected, bins=edges)[0] / len(expected)
    a = np.histogram(actual, bins=edges)[0] / len(actual)
    eps = 1e-6                       # avoid division by zero in empty bins
    e, a = np.clip(e, eps, None), np.clip(a, eps, None)
    return float(np.sum((a - e) * np.log(a / e)))


psi_scores = {col: population_stability_index(X_train[col], X_test[col])
              for col in X_train.select_dtypes(include=np.number).columns}
psi = (pd.Series(psi_scores).dropna().sort_values(ascending=False).to_frame("PSI"))
psi["status"] = pd.cut(psi["PSI"], [-np.inf, 0.10, 0.25, np.inf],
                       labels=["stable", "moderate shift", "major shift"])

# The model's own output is the most important thing to monitor.
train_scores_for_psi = reloaded.predict_proba(X_train)[:, 1]
score_psi = population_stability_index(train_scores_for_psi, test_scores)
print(f"PSI of the MODEL SCORE itself: {score_psi:.4f}")
print(f"  -> {'stable' if score_psi < 0.1 else 'moderate shift' if score_psi < 0.25 else 'MAJOR SHIFT'}\n")

display(psi.head(15).style.format({"PSI": "{:.4f}"}).set_caption(
    "Table 14 - Population stability index, training period vs test period"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))

top = psi.head(12).iloc[::-1]
colours = [C_AQUA if v < 0.10 else C_ORANGE if v < 0.25 else C_RED for v in top["PSI"]]
axes[0].barh(top.index, top["PSI"], color=colours, height=0.65)
axes[0].axvline(0.10, color=C_GRAY, ls="--", lw=1.5, label="0.10 investigate")
axes[0].axvline(0.25, color="#0b0b0b", ls="--", lw=1.5, label="0.25 retrain")
axes[0].set_xlabel("PSI (train vs test period)")
axes[0].set_title("Feature drift")
axes[0].legend(frameon=False, fontsize=9)

axes[1].hist(train_scores_for_psi, bins=40, density=True, alpha=0.55, color=C_BLUE,
             label="Training period")
axes[1].hist(test_scores, bins=40, density=True, alpha=0.55, color=C_ORANGE,
             label="Test period")
axes[1].set_xlabel("Predicted P(default)")
axes[1].set_ylabel("Density")
axes[1].set_title(f"Score distribution shift (PSI {score_psi:.3f})")
axes[1].legend(frameon=False, fontsize=9)

plt.tight_layout()
save_fig("18_psi_drift")

**Figure 18 / Table 14 — Feature drift (left) and score-distribution shift (right).**

Features above 0.25 have moved enough between the two periods that the model is being
applied to a population it was not trained on. In this dataset that is expected rather
than surprising — notebook 02 Figure 7 showed the platform's volume and mix changing
substantially across the window — but expected drift still degrades a model.

The right-hand panel is the one to watch in production, because it needs **no outcome
labels**. Default outcomes take months or years to materialise, so label-based monitoring
is always looking backwards. Score PSI is available the moment applications are scored,
which makes it the earliest warning available.

A caution on reading it: a stable score distribution does **not** prove the model still
works. The inputs can be unchanged while their relationship to default has shifted — the
concept drift that only outcomes reveal.

**Action →** Instrument score PSI as a daily production metric with alerting at 0.10 and
0.25. Compute feature PSI weekly. Both thresholds are conventions, not laws — calibrate
them against observed false-alarm rates once real traffic exists.

## 11.6 Performance decay and the retraining policy

Score PSI catches the inputs moving. Only outcomes reveal whether the *relationship* has
changed — so this tracks realised performance across the test period, quarter by quarter.
The slope is what sets the retraining cadence.

In [ ]:
test_meta = df.loc[~train_mask].reset_index(drop=True)
quarters = test_meta["issue_d"].dt.to_period("Q").astype(str)

rows = []
for q, idx in pd.Series(range(len(quarters)), index=quarters.values).groupby(level=0):
    pos = idx.to_numpy()
    yt = y_test.to_numpy()[pos]
    if len(pos) < 100 or len(np.unique(yt)) < 2:
        continue
    rows.append({"quarter": q, "n": len(pos), "default_rate": yt.mean(),
                 "ROC-AUC": roc_auc_score(yt, test_scores[pos]),
                 "PR-AUC": average_precision_score(yt, test_scores[pos])})
decay = pd.DataFrame(rows).set_index("quarter")

fig, ax = plt.subplots(figsize=(9.5, 4.4))
ax.plot(decay.index, decay["ROC-AUC"], color=C_BLUE, lw=2, marker="o", ms=6, label="ROC-AUC")
ax.axhline(final_roc, color=C_GRAY, ls="--", lw=1.5, label=f"overall {final_roc:.3f}")

if len(decay) >= 3:      # least-squares trend across the test window
    xs = np.arange(len(decay))
    slope, intercept = np.polyfit(xs, decay["ROC-AUC"], 1)
    ax.plot(decay.index, intercept + slope * xs, color=C_RED, ls=":", lw=2,
            label=f"trend {slope:+.4f} AUC/quarter")
    print(f"ROC-AUC trend: {slope:+.4f} per quarter ({slope * 4:+.4f} per year)")
    if slope < -0.005:
        print("-> Material decay: refit more often than annually.")
    else:
        print("-> No material decay across this window; an annual refit looks adequate.")

ax.set_ylabel("ROC-AUC")
ax.set_xlabel("Issue quarter")
ax.set_title("Performance decay across the test period")
ax.legend(frameon=False, fontsize=9)
plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
save_fig("19_performance_decay")
display(decay.style.format({"n": "{:,.0f}", "default_rate": "{:.3f}",
                            "ROC-AUC": "{:.3f}", "PR-AUC": "{:.3f}"})
        .set_caption("Table 15 - Realised performance by test quarter"))

**Figure 19 / Table 15 — Performance decay across the test period.**

The trend line is the retraining cadence, derived rather than guessed. A slope of
−0.005 AUC per quarter means roughly −0.02 per year, which would make an annual refit too
slow to hold the model at its evaluated performance.

Read the final quarters with the right-censoring caveat in mind: they contain a non-random
subsample of loans that resolved early, so some apparent movement there is the artefact
from notebook 02 Figure 7 rather than genuine decay.

**Action →** Set the refit cadence from the observed slope, and keep the last two quarters
out of the decision until they have matured.

### Retraining policy

| Trigger | Threshold | Response |
|---|---|---|
| **Score PSI** | > 0.25 | Investigate immediately; retrain if input drift is confirmed |
| **Feature PSI** on a top-importance feature | > 0.25 | Investigate the upstream data source first — often a pipeline change, not real drift |
| **ROC-AUC decay** on matured cohorts | > 0.02 below the model-card value | Retrain |
| **Calibration** (Brier / observed-vs-expected) | drift beyond tolerance | Recalibrate — cheaper than a full refit and often sufficient |
| **Scheduled** | Per the observed slope above | Refit on the most recent window regardless of alerts |

Two operational notes. **Recalibration is not retraining**: if ranking still holds but the
probabilities have drifted, refitting only the isotonic layer is far cheaper and lower-risk.
And every refit must **re-run notebooks 02–03 end to end**, not just `model.fit()` — the
split, the leakage checks and the fairness screen are all part of the deliverable, not
one-off setup.

## 11.7 The 6-question self-check

The process closes with six questions for any project. Answered here, with the section
that discharges each.

**1. Does my metric reflect the real cost of errors?**
Yes. The FN:FP = 4:1 cost matrix (NB01 §1.3) drives the operating threshold (NB03 §9.3),
and PR-AUC is primary because the expensive class is the rare one. The ratio itself is
still a placeholder — that is a stated limitation, not a hidden one.

**2. Does the split simulate real use?**
Yes. Out-of-time at 2016-01-01 (NB02 §4), because deployment scores future applicants.
NB03 §9.8 quantifies how much optimism a random split would have added. `TimeSeriesSplit`
is used for CV throughout, never shuffled folds.

**3. Are there features I would not have at prediction time?**
Checked column by column in the NB02 §3.2 data dictionary, blocked by `LEAKAGE_COLUMNS`,
and re-checked defensively in §3.7. The `known_at_application` column is the audit trail.

**4. Is every transform fitted on train only?**
Yes, and structurally rather than by discipline: all learned transforms live inside a
`Pipeline`, so `X_test` only ever reaches `.transform()`. Fixed domain bounds replace
percentile clipping, missing-rate screening is measured on training rows, and calibration
and threshold selection both use a time-ordered slice of training data.

**5. Do I have a baseline?**
Two (NB01 §2.5, computed NB03 §8.2). B0 majority class, and B1 `sub_grade` — **the
incumbent underwriting rule**. The lift over B1 is the project's actual claim.

**6. If the score looks too good, did I suspect it?**
Yes — this is the guard rail. The honest band on this dataset is ROC-AUC ≈ 0.68–0.72, and
anything near 0.99 means a leakage column survived. The band is documented in `CLAUDE.md`
and the leakage audit is re-run on every change.

---

## Deliverables produced

| Artifact | What it is |
|---|---|
| `artifacts/model.joblib` | The complete pipeline — preprocessing, model, calibration |
| `artifacts/model_card.json` | Provenance, metrics, versions, limitations |
| `artifacts/split_manifest.json` | The shared split, auditable and reproducible |
| `artifacts/metrics.csv` | Model comparison including both baselines |
| `artifacts/figures/` | Every figure, captioned and discussed |

**Not committed to git.** `data/` and `artifacts/` are git-ignored; they are reproduced by
running notebooks 02–04 in order.

In [ ]:
print("=" * 72)
print("STAGE D COMPLETE".center(72))
print("=" * 72)
print(f"Data source   : {DATA_SOURCE}")
if DATA_SOURCE == "synthetic":
    print("                ^ SMOKE TEST ONLY -- not findings. See README.md.")
print(f"Final model   : {FINAL_MODEL_NAME} + isotonic calibration")
print(f"OOT ROC-AUC   : {final_roc:.4f}")
print(f"OOT PR-AUC    : {final_pr:.4f}")
print(f"Score PSI     : {score_psi:.4f}")
print(f"Round-trip    : verified identical")
print(f"\nArtifacts in {ARTIFACTS}:")
for path in sorted(ARTIFACTS.iterdir()):
    if path.is_file():
        print(f"  {path.name:<24} {path.stat().st_size / 1024:>8.1f} KB")
print(f"  figures/{'':<16} {len(list(FIG_DIR.glob('*.png'))):>8} PNGs")